In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

#### Pivoting with method 1: median

In [3]:
df_piv_median = pd.read_csv('df_median.csv')
df_piv_median.head()

,id,time,variable,value,date
0,AS14.01,2014-02-26 13:00:00,mood,6.0,2014-02-26
1,AS14.01,2014-02-26 15:00:00,mood,6.0,2014-02-26
2,AS14.01,2014-02-26 18:00:00,mood,6.0,2014-02-26
3,AS14.01,2014-02-26 21:00:00,mood,7.0,2014-02-26
4,AS14.01,2014-02-27 09:00:00,mood,6.0,2014-02-27


In [3]:
ids = df_piv_median['id'].unique()
df_piv_median['date'] = pd.to_datetime(df_piv_median['date']) 

In [4]:
median_vars = ['mood', 'circumplex.arousal', 'circumplex.valence', 'activity']
sum_vars = [v for v in df_piv_median['variable'].unique() if v not in median_vars]  

median_group = df_piv_median[df_piv_median['variable'].isin(median_vars)].copy()
median_agg = (median_group.groupby(['id', 'date', 'variable'])['value'].median().reset_index())


sum_group = df_piv_median[df_piv_median['variable'].isin(sum_vars)].copy()
sum_agg = (sum_group.groupby(['id', 'date', 'variable'])['value'].sum().reset_index())

combined = pd.concat([median_agg, sum_agg], ignore_index=True)
daily_df_median = combined.pivot(index=['id', 'date'], columns='variable', values='value').reset_index()

daily_df_median.columns.name = None

daily_df_median.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,2.0
3,AS14.01,2014-02-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,3.0
4,AS14.01,2014-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [5]:
#set all missing values to 0 except of the scale or binary variables
all_columns = daily_df_median.columns.tolist()

cols_to_fill = [col for col in all_columns if col not in median_vars and col not in ['id', 'date']]

daily_df_median[cols_to_fill] = daily_df_median[cols_to_fill].fillna(0)
daily_df_median.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,0.0
1,AS14.01,2014-02-18,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,0.0,0.0
2,AS14.01,2014-02-19,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7.0,NaN,NaN,NaN,0.0,2.0
3,AS14.01,2014-02-20,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,3.0
4,AS14.01,2014-02-21,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.0


##### Missing days added in dataframe

In [6]:
all_dates = pd.date_range(daily_df_median['date'].min(), daily_df_median['date'].max())
full_index = pd.MultiIndex.from_product([ids, all_dates], names=['id', 'date'])
daily_df_median_full = daily_df_median.set_index(['id', 'date']).reindex(full_index).reset_index()
daily_df_median_full.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,0.0
1,AS14.01,2014-02-18,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,0.0,0.0
2,AS14.01,2014-02-19,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7.0,NaN,NaN,NaN,0.0,2.0
3,AS14.01,2014-02-20,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,3.0
4,AS14.01,2014-02-21,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.0


#### Pivoting with method 2: Kalman

In [4]:
df_piv_kalman = pd.read_csv('df_kalman.csv')
df_piv_kalman.head()

,id,time,variable,value,date
0,AS14.01,2014-02-26 13:00:00,mood,6.0,2014-02-26
1,AS14.01,2014-02-26 15:00:00,mood,6.0,2014-02-26
2,AS14.01,2014-02-26 18:00:00,mood,6.0,2014-02-26
3,AS14.01,2014-02-26 21:00:00,mood,7.0,2014-02-26
4,AS14.01,2014-02-27 09:00:00,mood,6.0,2014-02-27
